# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedwaqasahmad/FlyRank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Building the actual numeric feature vector for the churn classification task: encoding categorical fields, filling any missing values, and confirming the final matrix shape.

In [5]:
import os, subprocess
REPO_URL = "https://github.com/syedwaqasahmad/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_churned"] = df["trend_direction"].str.lower().eq("down").astype(int)

numeric_features = ["impressions_90d","search_volume","competition","cpc","word_count","char_count",
                     "days_with_impressions","days_with_sessions","content_age_days",
                     "days_since_last_update","ctr","avg_position","engagement_rate",
                     "scroll_rate","ai_traffic_pct"]
categorical_features = ["competition_level","content_type","main_intent","age_tier",
                         "freshness_tier","word_count_tier","impression_tier","position_tier"]

X_numeric = df[numeric_features].replace([np.inf, -np.inf], np.nan).fillna(0)
X_categorical = pd.get_dummies(df[categorical_features], dummy_na=True)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_churned"]

print("Feature matrix shape:", X.shape)
print("Label distribution:", y.value_counts().to_dict())

Feature matrix shape: (30000, 54)
Label distribution: {1: 16262, 0: 13738}


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

impressions_90d, sessions/clicks counts: observable, known before any decision — safe.
content_age_days, days_since_last_update: observable, safe, no missing values (verified in ML-04).
ctr, avg_position: observable ratios, safe — computed from historical search data, not from any future outcome.
Categorical tiers (age_tier, freshness_tier, etc.): pre-computed buckets of the numeric fields above — safe, but redundant with their numeric source.
All of the above exist BEFORE the label is determined — none of them require knowing the outcome first.

In [6]:
print("Missing values per numeric feature:")
print(X_numeric.isnull().sum()[X_numeric.isnull().sum() > 0])
print("\nAll numeric features exist independently of the label (trend_direction).")

Missing values per numeric feature:
Series([], dtype: int64)

All numeric features exist independently of the label (trend_direction).


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Attacking my own features: check what happens if trend_pct (the exact number the label is derived from) is added as a feature. This should produce a suspiciously perfect result — the signature of leakage.

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

clean_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X, y)
clean_auc = roc_auc_score(y, clean_tree.predict_proba(X)[:, 1])

X_leaky = X.copy()
X_leaky["trend_pct"] = df["trend_pct"].fillna(0)
leaky_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_leaky, y)
leaky_auc = roc_auc_score(y, leaky_tree.predict_proba(X_leaky)[:, 1])

print(f"Clean feature set ROC AUC: {clean_auc:.3f}")
print(f"Leaky feature set (with trend_pct) ROC AUC: {leaky_auc:.3f}")
print("\nThe leaky version jumps to near-perfect AUC — that's the signature of leakage.")

Clean feature set ROC AUC: 0.673
Leaky feature set (with trend_pct) ROC AUC: 1.000

The leaky version jumps to near-perfect AUC — that's the signature of leakage.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

trend_pct — excluded because it's the exact value trend_direction (my label) is computed from; including it teaches the model to copy the answer instead of finding a real signal (confirmed above).
Product decision flags (health_score, priority_score, action_type) — never shipped in this dataset by design.
client_id, content_id — join/context keys, not predictive signals.

In [8]:
excluded = {
    "trend_pct": "label-derived, direct leakage (confirmed above)",
    "product decision flags (health_score, priority_score, action_type)": "not shipped in this dataset by design",
    "client_id / content_id": "join keys, not predictive signals — used for context only"
}
for k, v in excluded.items():
    print(f"- {k}: {v}")

- trend_pct: label-derived, direct leakage (confirmed above)
- product decision flags (health_score, priority_score, action_type): not shipped in this dataset by design
- client_id / content_id: join keys, not predictive signals — used for context only


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.